# 12 — CatBoost: Ordered Boosting & Native Categoricals Without Target Leakage

**Difficulty:** ⭐⭐⭐⭐⭐ &nbsp;|&nbsp; **Estimated time:** 2.5 hours &nbsp;|&nbsp; **Topic:** Tree-Based Methods & Gradient Boosting

---

## Why This Matters

CatBoost (Prokhorenkova et al., Yandex 2018) solves a subtle but serious bug in how traditional gradient boosting handles **categorical features**: target leakage. Most boosting implementations quietly introduce this leakage when they encode categories using the target variable. CatBoost's ordered boosting is the first mathematically principled fix.

It also introduces **symmetric (oblivious) trees** — every node at the same depth uses the *same* split rule — which makes predictions fast and deterministic while providing strong built-in regularization.

---

## Real-World Analogy First

You are an appraiser estimating house #500 in the Downtown neighbourhood. To gauge what "Downtown" is worth, you want to know the **average price of all Downtown houses**.

A **naive appraiser** looks up the Downtown average including house #500 in the calculation — the very house you're trying to value. This is circular reasoning: you're using the house's own value to estimate its value. In small categories this causes massive overfitting because the model sees the answer embedded in its own input features.

A **CatBoost appraiser** follows a strict protocol: when appraising house #500, compute the Downtown average using only houses #1 through #499 — houses whose appraisals are already finalised and do not include #500. As more houses are processed the estimate stabilises and converges to the true Downtown premium, but it is **never contaminated by the sample being evaluated**. This is CatBoost's *ordered target statistics*.

---

## Learning Objectives

By the end of this notebook you will be able to:
- Explain why naive target encoding of categorical features causes target leakage
- Describe CatBoost's ordered target statistics and how they prevent leakage
- Use CatBoost's `cat_features` parameter for zero-preprocessing categorical handling
- Explain symmetric (oblivious) trees and their trade-offs
- Tune CatBoost's key hyperparameters (`depth`, `learning_rate`, `l2_leaf_reg`)
- Compare CatBoost, LightGBM, and XGBoost on accuracy, speed, and suitability

## Section 1 — Setup & Synthetic House Price Dataset

We build a dataset with **1000 houses** and 3 categorical features: `neighborhood` (5 categories), `house_type` (4 categories), and `condition` (3 categories). More categorical features than notebook 11 — this is where CatBoost shines.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.ensemble import GradientBoostingRegressor

np.random.seed(42)
sns.set_theme(style='whitegrid')

print('Libraries loaded.')

In [ ]:
# ── Install / import CatBoost, LightGBM, XGBoost with graceful fallbacks ─────
try:
    from catboost import CatBoostRegressor, Pool
    CATBOOST_AVAILABLE = True
    print('CatBoost available.')
except ImportError:
    CATBOOST_AVAILABLE = False
    print('CatBoost not installed. Falling back to sklearn GradientBoostingRegressor.')
    print('To install: pip install catboost')

try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
    print(f'LightGBM {lgb.__version__} available.')
except ImportError:
    LGBM_AVAILABLE = False
    print('LightGBM not installed — comparison section will skip it.')

try:
    import xgboost as xgb
    XGB_AVAILABLE = True
    print(f'XGBoost {xgb.__version__} available.')
except ImportError:
    XGB_AVAILABLE = False
    print('XGBoost not installed — comparison section will skip it.')

In [ ]:
# ── Generate synthetic house price dataset ────────────────────────────────────
np.random.seed(42)
N = 1000

neighborhoods = ['Downtown', 'Midtown', 'Uptown', 'Suburbs', 'Rural']
house_types   = ['Detached', 'Semi', 'Terrace', 'Flat']
conditions    = ['Excellent', 'Good', 'Fair']

neighborhood  = np.random.choice(neighborhoods, N)
house_type    = np.random.choice(house_types,   N)
condition     = np.random.choice(conditions,    N)

sqft          = np.random.normal(1800, 500, N).clip(600, 4000)
bedrooms      = np.random.randint(1, 7, N).astype(float)
bathrooms     = np.random.randint(1, 5, N).astype(float)
age_years     = np.random.randint(0, 80, N).astype(float)
lot_size      = np.random.normal(5000, 2000, N).clip(1000, 15000)
garage_spaces = np.random.randint(0, 4, N).astype(float)
dist_downtown = np.random.exponential(5, N).clip(0.5, 30)
school_rating = np.random.uniform(3, 10, N)

nbhd_premium  = {'Downtown': 60000, 'Midtown': 30000, 'Uptown': 45000,
                 'Suburbs': 10000,  'Rural': -20000}
type_premium  = {'Detached': 40000, 'Semi': 10000, 'Terrace': 0, 'Flat': -15000}
cond_premium  = {'Excellent': 25000, 'Good': 5000, 'Fair': -20000}

price = (
    120 * sqft
    + 15000 * bedrooms
    + 12000 * bathrooms
    -   800 * age_years
    +     8 * lot_size
    + 10000 * garage_spaces
    -  4000 * dist_downtown
    +  8000 * school_rating
    + np.array([nbhd_premium[n] for n in neighborhood])
    + np.array([type_premium[t] for t in house_type])
    + np.array([cond_premium[c] for c in condition])
    + np.random.normal(0, 30000, N)
).clip(50000, 1200000)

df = pd.DataFrame({
    'sqft':          sqft,
    'bedrooms':      bedrooms,
    'bathrooms':     bathrooms,
    'age_years':     age_years,
    'lot_size':      lot_size,
    'garage_spaces': garage_spaces,
    'dist_downtown': dist_downtown,
    'school_rating': school_rating,
    'neighborhood':  neighborhood,
    'house_type':    house_type,
    'condition':     condition,
    'price':         price
})

cat_cols = ['neighborhood', 'house_type', 'condition']
print(f'Dataset shape: {df.shape}')
print(f'Categorical features: {cat_cols}')
print(f'Price range:   ${df.price.min():,.0f} – ${df.price.max():,.0f}')
df.head()

In [ ]:
# ── Train / validation / test split ──────────────────────────────────────────
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f'Train : {X_train.shape[0]} samples')
print(f'Val   : {X_val.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')

# Encoded copies for sklearn / XGBoost
enc = OrdinalEncoder()
X_train_enc = X_train.copy()
X_val_enc   = X_val.copy()
X_test_enc  = X_test.copy()
X_train_enc[cat_cols] = enc.fit_transform(X_train[cat_cols])
X_val_enc[cat_cols]   = enc.transform(X_val[cat_cols])
X_test_enc[cat_cols]  = enc.transform(X_test[cat_cols])

# Cat feature indices for CatBoost
cat_feature_indices = [X_train.columns.get_loc(c) for c in cat_cols]
print(f'Cat feature indices: {cat_feature_indices}')

## Section 2 — The Categorical Feature Problem: Target Leakage

### Plain English: What Goes Wrong with Naive Target Encoding

**Naive mean encoding** replaces a category with the mean target value across all training samples in that category:

```
encode("Downtown") = mean(price for all Downtown houses in training set)
```

**The leakage problem:** when encoding house #500, its own price `y₅₀₀` contributes to `mean("Downtown")`. So the feature for house #500 partially *contains* `y₅₀₀`. The model sees the answer in its own input — this is **target leakage**.

**Consequence:** the model learns to trust the target-encoded feature too much. On training data it looks brilliant (it's literally cheating). On validation data where the encoding was computed on different samples, this advantage vanishes and the model underperforms.

### The Fix: Leave-One-Out Encoding (and why even that fails)

Leave-one-out encoding excludes the current sample when computing the mean:
```
encode_LOO(xᵢ) = mean(y for all j ≠ i in same category)
```
This removes *direct* leakage but **indirect leakage** remains: all samples in the category share information from each other, and the encoding at test time uses *training labels* in a biased way.

CatBoost's ordered approach completely solves both forms.

In [ ]:
# ── Demonstrate target leakage: train vs validation gap ───────────────────────
def naive_target_encode(X_tr, y_tr, X_te, cat_col, smoothing=0):
    """Full-data mean encoding (leaky for training set)."""
    global_mean = y_tr.mean()
    cat_means   = y_tr.groupby(X_tr[cat_col]).mean()
    if smoothing > 0:
        cat_counts = y_tr.groupby(X_tr[cat_col]).count()
        cat_means  = (cat_means * cat_counts + global_mean * smoothing) / (cat_counts + smoothing)
    encoded_te = X_te[cat_col].map(cat_means).fillna(global_mean)
    encoded_tr = X_tr[cat_col].map(cat_means).fillna(global_mean)
    return encoded_tr, encoded_te

def loo_target_encode(X_tr, y_tr, X_te, cat_col):
    """Leave-one-out encoding (reduces but doesn't eliminate leakage)."""
    df_tmp = X_tr[[cat_col]].copy()
    df_tmp['y'] = y_tr.values
    cat_sum   = df_tmp.groupby(cat_col)['y'].transform('sum')
    cat_count = df_tmp.groupby(cat_col)['y'].transform('count')
    loo_enc_tr = (cat_sum - df_tmp['y']) / (cat_count - 1)
    global_mean = y_tr.mean()
    cat_means_for_te = df_tmp.groupby(cat_col)['y'].mean()
    loo_enc_te = X_te[cat_col].map(cat_means_for_te).fillna(global_mean)
    return loo_enc_tr, loo_enc_te

# Build feature sets for each strategy
num_cols = [c for c in X_train.columns if c not in cat_cols]

def make_encoded_dataset(X_tr, y_tr, X_va, strategy='naive'):
    X_tr_out = X_tr[num_cols].copy().reset_index(drop=True)
    X_va_out = X_va[num_cols].copy().reset_index(drop=True)
    for col in cat_cols:
        if strategy == 'naive':
            enc_tr, enc_va = naive_target_encode(X_tr, y_tr, X_va, col)
        elif strategy == 'loo':
            enc_tr, enc_va = loo_target_encode(X_tr, y_tr, X_va, col)
        elif strategy == 'ordinal':
            le = LabelEncoder()
            enc_tr = pd.Series(le.fit_transform(X_tr[col]), index=X_tr.index)
            enc_va = pd.Series(le.transform(X_va[col]),   index=X_va.index)
        X_tr_out[col] = enc_tr.values
        X_va_out[col] = enc_va.values
    return X_tr_out, X_va_out

leakage_results = {}
for strategy in ['naive', 'loo', 'ordinal']:
    Xtr, Xva = make_encoded_dataset(X_train, y_train, X_val, strategy)
    from sklearn.ensemble import GradientBoostingRegressor as GBR
    m = GBR(n_estimators=150, max_depth=4, learning_rate=0.1, random_state=42)
    m.fit(Xtr, y_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, m.predict(Xtr)))
    val_rmse   = np.sqrt(mean_squared_error(y_val,   m.predict(Xva)))
    leakage_results[strategy] = {'train_rmse': train_rmse, 'val_rmse': val_rmse}

# Visualise train vs val gap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

strategies = list(leakage_results.keys())
labels     = ['Naive\nTarget Enc.', 'Leave-One-Out\nTarget Enc.', 'Ordinal\nEncoding']
train_rmses = [leakage_results[s]['train_rmse'] for s in strategies]
val_rmses   = [leakage_results[s]['val_rmse']   for s in strategies]
gaps        = [v - t for t, v in zip(train_rmses, val_rmses)]

x = np.arange(len(strategies))
w = 0.35
axes[0].bar(x - w/2, train_rmses, w, label='Train RMSE', color='#2196F3', edgecolor='black', linewidth=0.7)
axes[0].bar(x + w/2, val_rmses,   w, label='Val RMSE',   color='#F44336', edgecolor='black', linewidth=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, fontsize=9)
axes[0].set_title('Train vs Val RMSE by Encoding Strategy', fontsize=10, fontweight='bold')
axes[0].set_ylabel('RMSE ($)')
axes[0].legend()
for i, (tr, va) in enumerate(zip(train_rmses, val_rmses)):
    axes[0].text(i - w/2, tr + 200, f'${tr:,.0f}', ha='center', fontsize=8, color='#2196F3', fontweight='bold')
    axes[0].text(i + w/2, va + 200, f'${va:,.0f}', ha='center', fontsize=8, color='#F44336', fontweight='bold')

bar_colors = ['#F44336' if g > 5000 else '#4CAF50' for g in gaps]
axes[1].bar(labels, gaps, color=bar_colors, edgecolor='black', linewidth=0.7)
axes[1].set_title('Train–Val RMSE Gap (Generalisation Gap)\nRed = worse generalisation (leakage)', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Val RMSE − Train RMSE ($)')
axes[1].axhline(0, color='black', linewidth=0.8)
for i, g in enumerate(gaps):
    axes[1].text(i, g + 200, f'${g:,.0f}', ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Target Leakage Effect: Why Encoding Strategy Matters', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nConclusion:')
print('  Naive target encoding → large train–val gap (target leakage overfitting)')
print('  LOO encoding          → smaller but still present gap')
print('  Ordinal encoding      → no leakage but loses category order information')
print('  CatBoost ordered TS   → no leakage AND uses target information correctly')

## Section 3 — CatBoost's Solution: Ordered Target Statistics

### The Formula

For sample $x_i$ with categorical feature value $c$, CatBoost computes:

$$
\text{encode}(c, i) = \frac{\sum_{j < i} \mathbf{1}[c_j = c] \cdot y_j + a \cdot \bar{y}}{\sum_{j < i} \mathbf{1}[c_j = c] + a}
$$

Where:
- $j < i$ means only samples that appear **before** sample $i$ in a random permutation
- $\bar{y}$ is the global mean of $y$ (the **prior**)
- $a$ is a **smoothing parameter** (prevents noisy encodings for rare categories)

**Key properties:**
- When only 1 sample of category $c$ has been seen: encoding ≈ $\bar{y}$ (falls back to prior)
- As more samples accumulate: encoding converges toward the true category mean
- Sample $i$ **never** contributes to its own encoding — no leakage by construction
- Multiple random permutations are used (one per tree) to reduce variance

### Why Multiple Permutations?

Using a single fixed permutation would make the encoding for late-arriving samples (high $i$) much better than early samples (low $i$, less data available). CatBoost uses a **different random permutation for each tree**, so on average every sample benefits from the same expected number of predecessors.

In [ ]:
# ── Visualise Ordered Target Statistics: convergence ─────────────────────────
np.random.seed(42)

# Simulate ordered TS for the 'Downtown' category
# Extract all Downtown houses from training set
downtown_mask = X_train['neighborhood'] == 'Downtown'
downtown_prices = y_train[downtown_mask].values
n_downtown = len(downtown_prices)
true_mean  = downtown_prices.mean()
prior      = y_train.mean()
a          = 1.0  # smoothing

# Shuffle (simulate one permutation)
perm = np.random.permutation(n_downtown)
shuffled_prices = downtown_prices[perm]

# Compute ordered TS at each step
ordered_ts   = []
naive_ts     = []
cumsum        = 0.0
for i, price in enumerate(shuffled_prices):
    if i == 0:
        ordered_ts.append(prior)   # only prior available
    else:
        ordered_ts.append((cumsum + a * prior) / (i + a))
    cumsum += price
    # Naive uses all data including current
    naive_ts.append((cumsum + a * prior) / (i + 1 + a))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: ordered vs naive TS convergence
x_axis = np.arange(1, n_downtown + 1)
axes[0].plot(x_axis, naive_ts,   label='Naive TS (uses own label)', color='#F44336', linewidth=2)
axes[0].plot(x_axis, ordered_ts, label='Ordered TS (CatBoost)',     color='#2196F3', linewidth=2)
axes[0].axhline(true_mean, color='black', linestyle='--', linewidth=1.5, label=f'True Downtown mean (${true_mean:,.0f})')
axes[0].axhline(prior,     color='grey',  linestyle=':',  linewidth=1.5, label=f'Global prior (${prior:,.0f})')
axes[0].set_title('Ordered vs Naive Target Statistics\n(Downtown neighbourhood encoding)', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Number of Downtown samples seen so far')
axes[0].set_ylabel('Encoded value ($)')
axes[0].legend(fontsize=8)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Right: Show early-sample problem with naive TS (i=0 case)
# First house sees only itself (n=1) — big noise vs ordered which uses prior
naive_first_3  = naive_ts[:3]
ordered_first_3 = ordered_ts[:3]
cases = [f'House #{i+1}' for i in range(3)]
x3 = np.arange(3)
w  = 0.3
axes[1].bar(x3 - w/2, naive_first_3,   w, label='Naive TS',   color='#F44336', edgecolor='black')
axes[1].bar(x3 + w/2, ordered_first_3, w, label='Ordered TS', color='#2196F3', edgecolor='black')
axes[1].axhline(true_mean, color='black', linestyle='--', linewidth=1.5, label='True mean')
axes[1].axhline(prior, color='grey', linestyle=':', linewidth=1.5, label='Prior')
axes[1].set_xticks(x3)
axes[1].set_xticklabels(cases)
axes[1].set_title('Early Samples: Naive TS vs Ordered TS\n(Ordered falls back to prior; Naive noisy)', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Encoded value ($)')
axes[1].legend(fontsize=8)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('CatBoost Ordered Target Statistics: Convergence & Stability', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Downtown houses in train set: {n_downtown}')
print(f'True Downtown mean:   ${true_mean:,.0f}')
print(f'Global prior:         ${prior:,.0f}')
print(f'\nOrdered TS for house #1:  ${ordered_ts[0]:,.0f}  (only prior available)')
print(f'Ordered TS for house #5:  ${ordered_ts[4]:,.0f}  (4 predecessors)')
print(f'Ordered TS for house #{n_downtown}: ${ordered_ts[-1]:,.0f}  (converged toward true mean)')

## Section 4 — Native Categorical Handling with CatBoost

CatBoost requires zero preprocessing for categorical features. Just pass:
- String/object columns directly in the DataFrame, AND
- The column indices (or names) via `cat_features`

CatBoost internally handles encoding, ordered statistics, and finding optimal category splits — all without leakage.

In [ ]:
# ── CatBoost: native categorical vs ordinal encoding ─────────────────────────
cat_results = {}

if CATBOOST_AVAILABLE:
    # Version 1: CatBoost with native categorical features
    t0 = time.time()
    cb_native = CatBoostRegressor(
        iterations=200, depth=6, learning_rate=0.05,
        l2_leaf_reg=3, random_seed=42, verbose=0
    )
    # Pass cat_features as list of column names
    cb_native.fit(
        X_train, y_train,
        cat_features=cat_cols,
        eval_set=(X_val, y_val),
        early_stopping_rounds=30
    )
    cb_native_time = time.time() - t0
    cb_native_rmse = np.sqrt(mean_squared_error(y_val, cb_native.predict(X_val)))
    cat_results['CatBoost\nNative Cats'] = {'rmse': cb_native_rmse, 'time': cb_native_time}
    print(f'CatBoost (native cats) : time={cb_native_time:.2f}s  val RMSE=${cb_native_rmse:,.0f}')

    # Version 2: CatBoost with ordinal encoded features (ablation)
    t0 = time.time()
    cb_ordinal = CatBoostRegressor(
        iterations=200, depth=6, learning_rate=0.05,
        l2_leaf_reg=3, random_seed=42, verbose=0
    )
    cb_ordinal.fit(X_train_enc, y_train, eval_set=(X_val_enc, y_val), early_stopping_rounds=30)
    cb_ordinal_time = time.time() - t0
    cb_ordinal_rmse = np.sqrt(mean_squared_error(y_val, cb_ordinal.predict(X_val_enc)))
    cat_results['CatBoost\nOrdinal Enc.'] = {'rmse': cb_ordinal_rmse, 'time': cb_ordinal_time}
    print(f'CatBoost (ordinal enc) : time={cb_ordinal_time:.2f}s  val RMSE=${cb_ordinal_rmse:,.0f}')

# Version 3: sklearn GB with ordinal encoding (baseline)
t0 = time.time()
m_sklearn = GradientBoostingRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42
)
m_sklearn.fit(X_train_enc, y_train)
sklearn_time = time.time() - t0
sklearn_rmse = np.sqrt(mean_squared_error(y_val, m_sklearn.predict(X_val_enc)))
cat_results['sklearn GB\nOrdinal Enc.'] = {'rmse': sklearn_rmse, 'time': sklearn_time}
print(f'sklearn GB (ordinal)   : time={sklearn_time:.2f}s  val RMSE=${sklearn_rmse:,.0f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
names   = list(cat_results.keys())
rmses   = [cat_results[k]['rmse'] for k in names]
times   = [cat_results[k]['time'] for k in names]
colors  = ['#2ecc71', '#e74c3c', '#3498db'][:len(names)]

axes[0].bar(names, rmses, color=colors, edgecolor='black', linewidth=0.8)
axes[0].set_title('Validation RMSE by Method')
axes[0].set_ylabel('RMSE ($)')
for i, v in enumerate(rmses):
    axes[0].text(i, v + 100, f'${v:,.0f}', ha='center', fontweight='bold')

axes[1].bar(names, times, color=colors, edgecolor='black', linewidth=0.8)
axes[1].set_title('Training Time (seconds)')
axes[1].set_ylabel('Seconds')
for i, v in enumerate(times):
    axes[1].text(i, v + 0.02, f'{v:.2f}s', ha='center', fontweight='bold')

plt.suptitle('Native Categorical Handling: CatBoost vs Ordinal Encoding', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 5 — Symmetric (Oblivious) Trees

### Plain English

CatBoost builds **symmetric trees** (also called oblivious decision trees): at each depth level, **every node uses the same (feature, threshold) split**. A depth-3 symmetric tree has at most 8 leaves, and the path to any leaf is determined by 3 binary answers to the same 3 questions.

**Example for depth 3:**
```
Level 1: Is sqft > 1800?
Level 2: Is school_rating > 7.5?   ← same question at EVERY level-2 node
Level 3: Is age_years > 20?        ← same question at EVERY level-3 node
```
Every leaf is a combination of the 3 binary answers: `(sqft>1800, school>7.5, age>20)` → leaf value.

**What CatBoost gains from this:**
1. **Fast prediction:** path is determined by 3 feature lookups; leaf index = bitmask of answers
2. **Strong regularization:** fewer degrees of freedom → less overfitting
3. **Consistent feature importance:** same feature-threshold pair tested against all samples

**What CatBoost gives up:**
- Less expressive than asymmetric trees — can't model complex feature interactions that require different splits on different branches

In [ ]:
# ── Symmetric tree ASCII diagram ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

def draw_node(ax, x, y, text, color='#AED6F1', w=1.4, h=0.5, fontsize=8):
    rect = mpatches.FancyBboxPatch(
        (x - w/2, y - h/2), w, h,
        boxstyle='round,pad=0.05', facecolor=color,
        edgecolor='black', linewidth=1.0, zorder=2
    )
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize,
            fontweight='bold', zorder=3, wrap=True)

def draw_edge2(ax, x1, y1, x2, y2, label='', lc='black'):
    ax.annotate('', xy=(x2, y2 + 0.25), xytext=(x1, y1 - 0.25),
                arrowprops=dict(arrowstyle='->', color=lc, lw=1.2))
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    if label:
        ax.text(mx + 0.08, my, label, fontsize=7, color=lc)

# ── Asymmetric tree (left) ───────────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(-1, 9); ax.set_ylim(0, 7); ax.axis('off')
ax.set_title('Asymmetric Tree (XGBoost / LightGBM)\nDifferent split at every node', fontsize=10, fontweight='bold')

# Depth 0
draw_node(ax, 4.0, 6.3, 'sqft > 1800?',       '#AED6F1', w=1.6)
draw_edge2(ax, 3.3, 6.3, 1.5, 4.8, 'No',  '#d62728')
draw_edge2(ax, 4.7, 6.3, 6.5, 4.8, 'Yes', '#27ae60')
# Depth 1 — DIFFERENT splits
draw_node(ax, 1.5, 4.8, 'age > 40?',          '#F9E79F', w=1.4)
draw_node(ax, 6.5, 4.8, 'school > 7?',        '#F9E79F', w=1.4)
# Depth 2 — yet MORE different splits
draw_edge2(ax, 1.0, 4.8, 0.2, 3.3, 'No',  '#d62728')
draw_edge2(ax, 2.0, 4.8, 2.8, 3.3, 'Yes', '#27ae60')
draw_edge2(ax, 6.0, 4.8, 5.2, 3.3, 'No',  '#d62728')
draw_edge2(ax, 7.0, 4.8, 7.8, 3.3, 'Yes', '#27ae60')
draw_node(ax, 0.2, 3.3, 'lot>4000?',  '#A9DFBF', w=1.2, fontsize=7)
draw_node(ax, 2.8, 3.3, 'bath>2?',    '#A9DFBF', w=1.2, fontsize=7)
draw_node(ax, 5.2, 3.3, 'dist<3?',    '#A9DFBF', w=1.2, fontsize=7)
draw_node(ax, 7.8, 3.3, 'garage>1?',  '#A9DFBF', w=1.2, fontsize=7)
ax.text(4, 1.8, '← Different split question at every single node\n→ Very expressive, harder to regularise',
        ha='center', fontsize=8, style='italic',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

# ── Symmetric tree (right) ───────────────────────────────────────────────────
ax = axes[1]
ax.set_xlim(-1, 9); ax.set_ylim(0, 7); ax.axis('off')
ax.set_title('Symmetric Tree (CatBoost)\nSame split at every node of each depth', fontsize=10, fontweight='bold')

# Depth 0 — ONE split for the whole level
draw_node(ax, 4.0, 6.3, 'sqft > 1800?',       '#AED6F1', w=1.6)
ax.text(8.3, 6.3, '← Level 1\n   split', ha='left', fontsize=8, color='navy', style='italic')
draw_edge2(ax, 3.3, 6.3, 1.5, 4.8, 'No',  '#d62728')
draw_edge2(ax, 4.7, 6.3, 6.5, 4.8, 'Yes', '#27ae60')
# Depth 1 — SAME split at both nodes
draw_node(ax, 1.5, 4.8, 'school > 7?',  '#F9E79F', w=1.4)
draw_node(ax, 6.5, 4.8, 'school > 7?',  '#F9E79F', w=1.4)  # same!
ax.text(8.3, 4.8, '← Level 2\n   same!', ha='left', fontsize=8, color='navy', style='italic')
# Depth 2 — SAME split at all 4 nodes
for x_node in [0.2, 2.8, 5.2, 7.8]:
    draw_node(ax, x_node, 3.3, 'age > 20?', '#A9DFBF', w=1.2, fontsize=7)
ax.text(8.3, 3.3, '← Level 3\n   same!', ha='left', fontsize=8, color='navy', style='italic')
for prev_x, curr_x in [(1.0, 0.2), (2.0, 2.8), (6.0, 5.2), (7.0, 7.8)]:
    draw_edge2(ax, prev_x, 4.8, curr_x, 3.3)

ax.text(4, 1.8, '← Same split question at every node of a given depth\n→ Less expressive, but fast & well-regularised',
        ha='center', fontsize=8, style='italic',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow'))

plt.suptitle('Asymmetric vs Symmetric (Oblivious) Decision Trees', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 6 — Hyperparameter Tuning: `depth` × `learning_rate`

CatBoost's most important hyperparameters:

| Parameter | Role | Typical values |
|---|---|---|
| `iterations` | Number of trees | 100–2000 |
| `depth` | Tree depth (symmetric, so depth 6 = 64 leaves) | 4–8 |
| `learning_rate` | Step size (lower = slower but more accurate) | 0.01–0.1 |
| `l2_leaf_reg` | L2 regularisation on leaf values | 1–10 |
| `border_count` | Number of bins for numerical features | 32–255 |
| `random_strength` | Randomness for scoring splits | 0.1–1.0 |

**Note:** because CatBoost uses symmetric trees, `depth=6` means exactly `2^6 = 64` leaves. This is much more controlled than LightGBM's leaf-wise `num_leaves`.

In [ ]:
# ── Hyperparameter scan: depth × learning_rate ────────────────────────────────
depths        = [4, 6, 8]
learning_rates = [0.01, 0.05, 0.10]

heatmap_data = np.zeros((len(learning_rates), len(depths)))

for i, lr in enumerate(learning_rates):
    for j, d in enumerate(depths):
        if CATBOOST_AVAILABLE:
            m = CatBoostRegressor(
                iterations=200, depth=d, learning_rate=lr,
                l2_leaf_reg=3, random_seed=42, verbose=0
            )
            m.fit(X_train, y_train, cat_features=cat_cols)
            preds = m.predict(X_val)
        else:
            m = GradientBoostingRegressor(
                n_estimators=200, max_depth=d, learning_rate=lr,
                random_state=42
            )
            m.fit(X_train_enc, y_train)
            preds = m.predict(X_val_enc)
        heatmap_data[i, j] = np.sqrt(mean_squared_error(y_val, preds))

heatmap_df = pd.DataFrame(
    heatmap_data,
    index=[f'lr={v}' for v in learning_rates],
    columns=[f'depth={v}' for v in depths]
)

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(heatmap_df, annot=True, fmt='.0f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Val RMSE ($)'})
ax.set_title('CatBoost Validation RMSE ($)\ndepth × learning_rate  (green = lower = better)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Tree depth')
ax.set_ylabel('Learning rate')
plt.tight_layout()
plt.show()

best_idx = np.unravel_index(heatmap_data.argmin(), heatmap_data.shape)
print(f'\nBest combination:')
print(f'  depth         = {depths[best_idx[1]]}')
print(f'  learning_rate = {learning_rates[best_idx[0]]}')
print(f'  val RMSE      = ${heatmap_data[best_idx]:,.0f}')
print('\nPattern: very low learning_rate + small depth → underfitting (top-left)')
print('         very high depth + high lr  → possible overfitting (bottom-right)')

## Section 7 — Three-Way Comparison: CatBoost vs LightGBM vs XGBoost

We compare all three frameworks on the same dataset: identical number of iterations, similar learning rates. This gives a fair picture of the trade-offs.

In [ ]:
# ── Three-way comparison ──────────────────────────────────────────────────────
comparison = {}

# 1. CatBoost
if CATBOOST_AVAILABLE:
    t0 = time.time()
    m_cb = CatBoostRegressor(
        iterations=300, depth=6, learning_rate=0.05,
        l2_leaf_reg=3, random_seed=42, verbose=0
    )
    m_cb.fit(X_train, y_train, cat_features=cat_cols)
    comparison['CatBoost'] = {
        'time': time.time() - t0,
        'val_rmse': np.sqrt(mean_squared_error(y_val, m_cb.predict(X_val))),
        'color': '#e74c3c',
        'note': 'Native cats, symmetric trees'
    }
    print(f"CatBoost   time={comparison['CatBoost']['time']:.2f}s  "
          f"val RMSE=${comparison['CatBoost']['val_rmse']:,.0f}")

# 2. LightGBM
if LGBM_AVAILABLE:
    t0 = time.time()
    m_lgbm = lgb.LGBMRegressor(
        n_estimators=300, num_leaves=63, learning_rate=0.05,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
        random_state=42, verbose=-1
    )
    m_lgbm.fit(X_train, y_train, categorical_feature=cat_cols)
    comparison['LightGBM'] = {
        'time': time.time() - t0,
        'val_rmse': np.sqrt(mean_squared_error(y_val, m_lgbm.predict(X_val))),
        'color': '#2ecc71',
        'note': 'Histogram, leaf-wise'
    }
    print(f"LightGBM   time={comparison['LightGBM']['time']:.2f}s  "
          f"val RMSE=${comparison['LightGBM']['val_rmse']:,.0f}")

# 3. XGBoost
if XGB_AVAILABLE:
    t0 = time.time()
    m_xgb = xgb.XGBRegressor(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=3,
        random_state=42, verbosity=0, eval_metric='rmse'
    )
    m_xgb.fit(X_train_enc, y_train, eval_set=[(X_val_enc, y_val)], verbose=False)
    comparison['XGBoost'] = {
        'time': time.time() - t0,
        'val_rmse': np.sqrt(mean_squared_error(y_val, m_xgb.predict(X_val_enc))),
        'color': '#3498db',
        'note': 'Sorted splits, level-wise'
    }
    print(f"XGBoost    time={comparison['XGBoost']['time']:.2f}s  "
          f"val RMSE=${comparison['XGBoost']['val_rmse']:,.0f}")

# 4. sklearn GB (fallback baseline always computed)
t0 = time.time()
m_sk = GradientBoostingRegressor(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=42
)
m_sk.fit(X_train_enc, y_train)
comparison['sklearn GB'] = {
    'time': time.time() - t0,
    'val_rmse': np.sqrt(mean_squared_error(y_val, m_sk.predict(X_val_enc))),
    'color': '#95a5a6',
    'note': 'Reference baseline'
}
print(f"sklearn GB time={comparison['sklearn GB']['time']:.2f}s  "
      f"val RMSE=${comparison['sklearn GB']['val_rmse']:,.0f}")

# Results table
print(f'\n{"Model":<14} {"Time (s)":>10} {"Val RMSE ($)":>14} {"Notes"}')
print('-' * 65)
for name, info in comparison.items():
    print(f'{name:<14} {info["time"]:>10.2f} {info["val_rmse"]:>14,.0f}   {info["note"]}')

In [ ]:
# ── Comparison charts ─────────────────────────────────────────────────────────
if len(comparison) < 2:
    print('Not enough models to plot — install at least one of catboost/lightgbm/xgboost.')
else:
    names  = list(comparison.keys())
    colors = [comparison[k]['color'] for k in names]
    times  = [comparison[k]['time']     for k in names]
    rmses  = [comparison[k]['val_rmse'] for k in names]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].bar(names, times, color=colors, edgecolor='black', linewidth=0.8)
    axes[0].set_title('Training Time (300 trees, 800 training samples)')
    axes[0].set_ylabel('Seconds')
    for i, v in enumerate(times):
        axes[0].text(i, v + 0.02, f'{v:.2f}s', ha='center', fontweight='bold', fontsize=9)

    axes[1].bar(names, rmses, color=colors, edgecolor='black', linewidth=0.8)
    axes[1].set_title('Validation RMSE ($)')
    axes[1].set_ylabel('RMSE ($)')
    for i, v in enumerate(rmses):
        axes[1].text(i, v + 100, f'${v:,.0f}', ha='center', fontweight='bold', fontsize=9)

    plt.suptitle('Three-Way Comparison: CatBoost vs LightGBM vs XGBoost vs sklearn GB\n'
                 '(same dataset, same iterations, similar hyperparameters)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Section 8 — CatBoost Feature Importance

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────────
if CATBOOST_AVAILABLE:
    fi = pd.Series(
        m_cb.get_feature_importance(),
        index=X_train.columns
    ).sort_values(ascending=True)

    # Colour categoricals differently
    bar_colors = ['#e74c3c' if f in cat_cols else '#3498db' for f in fi.index]

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = fi.plot(kind='barh', ax=ax, color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.set_title('CatBoost Feature Importance\n(red = categorical features)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance score (%)')

    legend_elements = [
        mpatches.Patch(color='#e74c3c', label='Categorical feature'),
        mpatches.Patch(color='#3498db', label='Numerical feature')
    ]
    ax.legend(handles=legend_elements, loc='lower right')
    plt.tight_layout()
    plt.show()

    print('Top 3 features:')
    for feat, imp in fi.sort_values(ascending=False).head(3).items():
        tag = '(categorical)' if feat in cat_cols else '(numerical)'
        print(f'  {feat:<20} {imp:.1f}%  {tag}')
else:
    # Fallback: sklearn feature importance
    fi = pd.Series(
        m_sk.feature_importances_,
        index=X_train_enc.columns
    ).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 5))
    fi.plot(kind='barh', ax=ax, color='#95a5a6', edgecolor='black', linewidth=0.5)
    ax.set_title('sklearn GB Feature Importance (CatBoost not available)', fontsize=11, fontweight='bold')
    ax.set_xlabel('Importance score')
    plt.tight_layout()
    plt.show()

## Section 9 — Framework Reference: CatBoost vs LightGBM vs XGBoost

In [ ]:
# ── Reference comparison table ────────────────────────────────────────────────
table = pd.DataFrame({
    'Aspect': [
        'Categorical features', 'Target leakage', 'Tree shape',
        'Training speed', 'GPU support', 'Default depth param',
        'Regularisation', 'Best use case'
    ],
    'CatBoost': [
        'Native (ordered TS)', 'Prevented by design', 'Symmetric (oblivious)',
        'Slowest', 'Yes', 'depth (4–8)',
        'Built-in + l2_leaf_reg', 'Many categoricals, preventing leakage'
    ],
    'LightGBM': [
        'Native (basic partition)', 'Possible', 'Asymmetric, leaf-wise',
        'Fastest', 'Yes', 'num_leaves (31+)',
        'Histogram + GOSS', 'Large datasets, speed-critical'
    ],
    'XGBoost': [
        'Must encode', 'Possible', 'Asymmetric, level-wise',
        'Fast', 'Yes', 'max_depth (6)',
        'L1/L2 on weights', 'General purpose, robust'
    ],
    'sklearn GB': [
        'Must encode', 'Possible', 'Asymmetric, level-wise',
        'Slowest', 'No', 'max_depth (3)',
        'L1/L2 via min_samples', 'Reference, small datasets'
    ]
})
table.set_index('Aspect', inplace=True)
print('Framework Comparison Table:')
print(table.to_string())

## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** Why does using target-encoded categories on the full training set cause target leakage? Give a concrete example with 3 "Downtown" houses.

<details>
<summary>Show answer</summary>

Suppose the Downtown category contains exactly 3 houses with prices `[$300k, $350k, $400k]`.

Naive mean encoding computes:
```
encode("Downtown") = mean([300k, 350k, 400k]) = $350k
```
This encoding is then assigned as a feature to **all three houses**, including the one priced at $350k. When the model trains on house #2 (price = $350k), its "neighborhood" feature already says $350k — the model has essentially been handed the answer. It can give this feature very high weight and achieve near-zero training error.

On the validation set, however, no Downtown house was in the training set's mean (or different houses were). The encoding no longer reflects that specific house's price. The model relied on a shortcut (the leaky feature) that does not exist at test time, leading to poor generalisation.

With **ordered target statistics**, when encoding house #2, only houses #1 is used: `encode = (300k + a×prior) / (1 + a)`. House #2's own price $350k is never part of its own encoding.

</details>

---

**Q2.** CatBoost's symmetric trees use the same split at each depth level. This is less expressive than XGBoost's asymmetric trees. What does CatBoost get in return?

<details>
<summary>Show answer</summary>

CatBoost gains three concrete benefits from symmetric (oblivious) trees:

1. **Fast prediction:** since every level uses one (feature, threshold) pair, the leaf index is simply a bitmask of `depth` binary answers. A depth-6 tree needs only 6 comparisons and a table lookup — no traversal of a branching structure. This makes inference very fast, especially on CPUs.

2. **Built-in regularisation:** a symmetric tree has far fewer degrees of freedom than an asymmetric tree of the same depth. At depth 6, a symmetric tree chooses 6 (feature, threshold) pairs; an asymmetric tree can choose a different pair at each of the 63 internal nodes. The constrained structure acts like a strong prior against overfitting, particularly beneficial on small or noisy datasets.

3. **Consistent feature importance:** since the same split is applied to all samples at a given depth, the model evaluates feature usefulness uniformly across the entire data distribution, rather than specialising splits to sub-regions. This makes feature importances more interpretable and stable.

</details>

---

**Q3.** CatBoost is generally slower to train than LightGBM. Why would you still choose CatBoost?

<details>
<summary>Show answer</summary>

Several practical reasons favour CatBoost despite its slower speed:

1. **High-cardinality categorical features:** if your dataset has many categorical columns with many categories (e.g., zip codes, product IDs, user IDs), CatBoost's ordered target statistics handle them correctly without the risk of target leakage that plagues naive encodings. This is its biggest qualitative advantage — not just a speed/accuracy trade-off but a correctness guarantee.

2. **Small-to-medium datasets:** symmetric trees are more regularised, so CatBoost tends to overfit less on small datasets (< 10k rows). LightGBM's leaf-wise growth can overfit aggressively with small data even with `min_data_in_leaf` set.

3. **Minimal preprocessing:** CatBoost works out of the box with string categorical columns, missing values, and heterogeneous feature types — no encoding pipeline needed. This reduces code complexity and the risk of preprocessing bugs (e.g., train/test leakage in encoders).

4. **Training is one-time; inference is repeated:** if a model will be called for predictions millions of times, CatBoost's fast symmetric-tree inference may outweigh its slower training. The one-time training cost is paid once; the inference speedup is paid every request.

5. **Reproducibility:** CatBoost's training is fully deterministic with a fixed `random_seed`, making experiments easier to reproduce — particularly important in regulated industries.

</details>

## Summary

| Concept | Key Idea | Why It Matters |
|---|---|---|
| Target leakage | Using sample's own label to encode its features | Inflates train accuracy; collapses on val/test |
| Ordered target statistics | Encode category using only previous samples in permutation | Eliminates leakage by construction |
| Smoothing parameter `a` | Blends category mean with global prior | Prevents noisy encodings for rare categories |
| Multiple permutations | Different permutation per tree | Ensures all samples have equal expected predecessors |
| Symmetric trees | Same split at every node of a depth level | Fast inference, strong regularisation |
| `depth` parameter | Controls tree depth (2^depth leaves) | Primary complexity control; 6 → 64 leaves |
| `l2_leaf_reg` | L2 penalty on leaf values | Prevents extreme leaf values |

**When to use CatBoost:**
- Your dataset has many categorical features, especially high-cardinality ones
- You want to avoid encoding pipelines entirely
- You need correctness guarantees around target leakage
- Small-to-medium datasets where regularisation matters
- Fast inference matters more than fast training

**When to prefer LightGBM:**
- Very large datasets (1M+ rows) where training speed is critical
- You already have well-encoded features
- Memory is constrained

**Decision heuristic:**
```
Many string categoricals → CatBoost first
Large dataset, speed critical → LightGBM first  
General purpose, robust baseline → XGBoost first
```

**Week 9 complete.** You now have a thorough understanding of the full gradient boosting family: from CART and sklearn GB (notebooks 1–3) through XGBoost, LightGBM, and CatBoost (notebooks 10–12).